In [ ]:
from PyCROSL.CRO_SL import *
from PyCROSL.AbsObjectiveFunc import *
from PyCROSL.SubstrateReal import *
from PyCROSL.SubstrateInt import *

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import os




"""'''
File names to store the solutions provided by the algorithm
"""

filename = 'Test_Paper_1'
path_output = './Results/Test_Paper/'
# Create directory
if not os.path.exists(path_output):
    os.makedirs(path_output)

"""
Path and name of the predictor dataset and target dataset
"""
path_input = './Data/Paper/'
predictor_file = 'predictors_dataset.csv'
target_file = 'target.csv'



# Load the dataset
pred_dataframe = pd.read_csv(path_input+predictor_file, index_col=0)
pred_dataframe.index = pd.to_datetime(pred_dataframe.index)
target_dataset = pd.read_csv(path_input+target_file, index_col=0)
target_dataset.index = pd.to_datetime(target_dataset.index)


# Create an empty file to store the solutions provided by the algorithm
sol_data = pd.DataFrame(columns=['CV','Test','Sol'])

indiv_file = path_output+filename+'.csv'
solution_file = 'CRO_LogReg_'+filename+'.csv'
sol_data.to_csv(indiv_file,sep=' ',header=sol_data.columns,index=None)


# LAG and WINDOW
MAX_LAG = 180
MAX_WINDOW = 60
MAX_SHIFT = MAX_LAG + MAX_WINDOW
NLEN = len(pred_dataframe)
idx_pred = pred_dataframe.index[MAX_SHIFT:]
valid_idx = idx_pred.intersection(target_dataset.index)

X_blocks = []
col_meta = []
col_index = {} # To identify lags
col_id = 0
for var_i, col in enumerate(pred_dataframe.columns):
    s = pred_dataframe[col].to_numpy()
    for lag in range(1, MAX_SHIFT + 1):
        xlag = s[MAX_SHIFT - lag : NLEN - lag]
        X_blocks.append(xlag.reshape(-1, 1))
        col_meta.append((var_i, lag))
        col_index[(var_i, lag)] = col_id 
        col_id += 1

X_full = np.hstack(X_blocks).astype(np.float32)
pos = idx_pred.get_indexer(valid_idx)
X_full = X_full[pos, :]

y_full = target_dataset.reindex(valid_idx)['Target'].to_numpy()

print(X_full.shape)
print(y_full.shape)
print(target_dataset.shape)

(8859, 17040)
(8859,)
(10209, 1)


In [47]:
X_full

array([[-2.0620264e+02,  3.1224390e+02,  8.7153107e+02, ...,
         3.0000000e+00,  2.0000000e+00,  1.0000000e+00],
       [-4.9133365e+02, -2.0620264e+02,  3.1224390e+02, ...,
         4.0000000e+00,  3.0000000e+00,  2.0000000e+00],
       [-8.5784253e+02, -4.9133365e+02, -2.0620264e+02, ...,
         5.0000000e+00,  4.0000000e+00,  3.0000000e+00],
       ...,
       [ 5.2160547e+02,  5.1655396e+02,  1.0566499e+03, ...,
         3.0000000e+00,  2.0000000e+00,  1.0000000e+00],
       [ 1.0408044e+03,  5.2160547e+02,  5.1655396e+02, ...,
         4.0000000e+00,  3.0000000e+00,  2.0000000e+00],
       [ 1.5525731e+03,  1.0408044e+03,  5.2160547e+02, ...,
         5.0000000e+00,  4.0000000e+00,  3.0000000e+00]],
      shape=(8859, 17040), dtype=float32)

In [40]:

def solution_to_selected_cols(solution, p, col_index, max_shift):
    time_sequences = np.array(solution[:p]).astype(int)
    time_lags      = np.array(solution[p:2*p]).astype(int)
    variable_sel   = np.array(solution[2*p:3*p]).astype(int)

    selected_cols = []
    for i in range(p):
        if variable_sel[i] == 0:
            continue
        win = int(time_sequences[i])
        if win <= 0:
            continue
        start = int(time_lags[i])
        for j in range(win):
            lag = start + j
            if 1 <= lag <= max_shift:
                selected_cols.append(col_index[(i, lag)])
    return selected_cols

In [46]:
import numpy as np

# --- parámetros simulados ---
p = 4                # número de variables
MAX_LAG = 10
MAX_WINDOW = 5
MAX_SHIFT = MAX_LAG + MAX_WINDOW - 1

# --- col_index simulado ---
col_index = {}
col_id = 0
for i in range(p):
    for lag in range(1, MAX_SHIFT + 1):
        col_index[(i, lag)] = col_id
        col_id += 1

# --- función (la misma que usarás luego) ---
def solution_to_selected_cols(solution, p, col_index, max_shift):
    time_sequences = np.array(solution[:p]).astype(int)
    time_lags      = np.array(solution[p:2*p]).astype(int)
    variable_sel   = np.array(solution[2*p:3*p]).astype(int)

    selected_cols = []
    for i in range(p):
        if variable_sel[i] == 0:
            continue
        seq = time_sequences[i]
        if seq <= 0:
            continue
        start = time_lags[i]
        for j in range(seq):
            lag = start + j
            if 1 <= lag <= max_shift:
                selected_cols.append(col_index[(i, lag)])
    return selected_cols

# --- solution de prueba ---
solution = np.array([
    2, 0, 3, 1,      # time_sequences
    3, 5, 1, 7,      # time_lags
    1, 1, 0, 1       # variable_selection
])

# --- ejecutar ---
selected_cols = solution_to_selected_cols(solution, p, col_index, MAX_SHIFT)

print("selected_cols:", selected_cols)
print("n_cols:", len(selected_cols))

selected_cols: [2, 3, 48]
n_cols: 3


In [45]:
cols = solution_to_selected_cols(solution, p, col_index, 2)
cols

[7681]